# `pheno_01_cdr_inventory` — every phenotype we could test, at a 50-sample floor

Run in a **Verily Workbench** Jupyter app with CDR BigQuery access.

Companion to `../terra/pheno_00_table_summary.ipynb`, which summarises the
PhecodeX table the FELIX scan is actually fitted on. That notebook answers
*how good are the phecodes we test*. This one answers the prior question:
**what else is in the CDR that clears the same floor**, across the phenotype
families *All by All* used (medRxiv 2026.05.08.26350964) — 1,723 PhecodeX
diseases, 1,600 ATC prescriptions, 139 self-reported survey diseases, 85 lab
measurements, 10 physical measurements at their 392,030-participant scale.

**What it does**

1. Reads the long-read analysis cohort (`analysis_samples.txt`, one id per line).
2. Aggregates four CDR domains **inside BigQuery**, restricted to that cohort —
   no person-level rows are pulled down:
   - `drug_exposure` to ATC 3rd/4th/5th-level classes (binary exposures)
   - `measurement` to quantitative labs and physical measurements
   - `observation` to survey question x answer pairs (binary self-report)
   - `condition_occurrence` to SNOMED condition concepts (optional, off by
     default; overlaps the phecodes)
3. Reads the existing PhecodeX table if it is reachable, so the inventory is
   complete rather than only the new families.
4. Applies one testability rule per trait type and writes a single tidy table.

**Testability rule** (`MIN_SAMPLES`, default 50)

| Trait type | Testable when |
|---|---|
| binary | `min(n_cases, n_eligible - n_cases) >= MIN_SAMPLES` |
| quantitative | `n_measured >= MIN_SAMPLES` and `n_distinct_values >= MIN_DISTINCT` |

The binary rule uses the **smaller arm**, which is what drives power — the same
convention as `scripts/select_phenotypes.py`. `n_eff` is reported alongside
(`4 * n_cases * (1 - phi)` for binary, `n_measured` for quantitative) so families
can be compared on information rather than on trait counts.

**Outputs** (under `~/pheno_inventory/` by default)

| File | Contents |
|---|---|
| `phenotype_inventory.tsv` | one row per candidate phenotype, all families |
| `phenotype_inventory_export.tsv` | same, counts below `MIN_CELL` blanked |
| `inventory_by_family.tsv` | candidates / testable / median n_eff per family |
| `inventory_cutoff_curve.tsv` | testable count vs floor, 1-1000, per family |
| `inventory_units_audit.tsv` | unit heterogeneity per measurement concept |
| `phenotype_inventory.png` | family counts + cutoff curves |
| `summary.md` | headline numbers |

**This notebook is read-only against the CDR.** Every query is dry-run costed
first and refused above `MAX_QUERY_GB`.

## 1. Config — edit these

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

# ---- cohort and floor -------------------------------------------------------
# One research/person id per line. select_phenotypes.py writes analysis_samples.txt.
COHORT_FILE = os.environ.get("INV_COHORT_FILE", "")   # blank -> probe the candidates below
COHORT_CANDIDATES = [
    Path("analysis_samples.txt"),
    Path.home() / "analysis_samples.txt",
    Path("pheno_selection/analysis_samples.txt"),
    Path.home() / "pheno_selection" / "analysis_samples.txt",
]
# gs:// prefix holding analysis_samples.txt, if the cohort is only on a bucket.
COHORT_GCS_PREFIX = os.environ.get("INV_COHORT_GCS", "").rstrip("/")

MIN_SAMPLES = int(os.environ.get("INV_MIN_SAMPLES", "50"))   # the floor in the question
MIN_DISTINCT = int(os.environ.get("INV_MIN_DISTINCT", "10")) # degenerate-measurement guard
MIN_CELL = int(os.environ.get("INV_MIN_CELL", "20"))         # export suppression floor

# ---- which domains to inventory ---------------------------------------------
DO_PHECODES = True       # existing PhecodeX table, if reachable
DO_DRUGS = True          # drug_exposure -> ATC
DO_MEASUREMENTS = True   # measurement -> labs + physical measurements
DO_SURVEYS = True        # observation -> survey question x answer
DO_CONDITIONS = False    # condition_occurrence -> SNOMED; overlaps the phecodes

ATC_LEVELS = ("ATC 3rd", "ATC 4th", "ATC 5th")   # All by All's A10BJ example is ATC 4th

# ---- the existing PhecodeX table (optional) ---------------------------------
# Local path or gs:// URI. The Terra copy lives on the AoU RW bucket and is not
# normally reachable from Verily Workbench; leave blank there and the phecode
# rows are skipped with a note rather than failing.
PHENOTYPE_CSV = os.environ.get("INV_PHENOTYPE_CSV", "")

# ---- BigQuery ---------------------------------------------------------------
CDR_OVERRIDE = ""                                            # blank -> $WORKSPACE_CDR
MAX_QUERY_GB = float(os.environ.get("INV_MAX_QUERY_GB", "600"))  # refuse bigger scans
ON_DEMAND_USD_PER_TIB = 6.25                                 # for the printed estimate only

OUT_DIR = Path(os.environ.get("INV_OUT_DIR", str(Path.home() / "pheno_inventory")))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR :", OUT_DIR)
print("floor   :", MIN_SAMPLES, "in the smaller arm (binary) / measured (quantitative)")
print("domains :", ", ".join(n for n, on in [
    ("phecodes", DO_PHECODES), ("drugs", DO_DRUGS), ("measurements", DO_MEASUREMENTS),
    ("surveys", DO_SURVEYS), ("conditions", DO_CONDITIONS)] if on))

## 2. CDR, project, cohort

In [ ]:
def read_ids(path: Path) -> list[int]:
    raw = [s.strip() for s in path.read_text().splitlines() if s.strip()]
    bad = [s for s in raw if not re.fullmatch(r"\d+", s)]
    if bad:
        raise ValueError(f"{path}: {len(bad)} ids are not integers, e.g. {bad[:3]}")
    ids = sorted({int(s) for s in raw})
    if len(ids) != len(raw):
        print(f"note: {len(raw) - len(ids)} duplicate ids collapsed")
    return ids


def resolve_cdr() -> str:
    if CDR_OVERRIDE.strip():
        return CDR_OVERRIDE.strip()
    value = (os.environ.get("WORKSPACE_CDR") or "").strip()
    if not value:
        raise RuntimeError("WORKSPACE_CDR is unset. Set CDR_OVERRIDE to project.dataset.")
    return value


def resolve_cohort() -> tuple[list[int], Path]:
    """The analysis cohort as INT64 person ids, plus where it came from."""
    if COHORT_FILE.strip():
        p = Path(COHORT_FILE).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f"INV_COHORT_FILE={p} does not exist")
        return read_ids(p), p.resolve()
    for p in COHORT_CANDIDATES:
        if p.is_file():
            return read_ids(p), p.resolve()
    if COHORT_GCS_PREFIX:
        dest = OUT_DIR / "analysis_samples.txt"
        if not dest.exists():
            subprocess.check_call(
                ["gsutil", "cp", f"{COHORT_GCS_PREFIX}/analysis_samples.txt", str(dest)])
        return read_ids(dest), dest.resolve()
    raise FileNotFoundError(
        "No cohort list found. Set INV_COHORT_FILE (or COHORT_FILE) to the long-read\n"
        "analysis_samples.txt, or INV_COHORT_GCS to the bucket prefix holding it.\n"
        "Probed:\n  " + "\n  ".join(str(p) for p in COHORT_CANDIDATES))


CDR = resolve_cdr()
PROJECT = (os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GOOGLE_PROJECT")
           or os.environ.get("GCP_PROJECT") or None)
client = bigquery.Client(project=PROJECT)
COHORT_IDS, COHORT_SRC = resolve_cohort()
N = len(COHORT_IDS)

assert N >= MIN_SAMPLES, f"cohort of {N} is smaller than the {MIN_SAMPLES}-sample floor"
print("CDR     :", CDR)
print("project :", PROJECT or client.project)
print("cohort  :", COHORT_SRC)
print(f"n       : {N:,} participants")

## 3. Schema probe and the costed query helper

Every CDR query below is restricted to the cohort with an `ARRAY<INT64>`
parameter and aggregated server-side, so nothing person-level is returned. The
dry run reports the scan before the real query runs; `MAX_QUERY_GB` is the stop.
`measurement` is the largest table in the CDR — expect that query to dominate.

In [ ]:
def table_exists(table: str) -> bool:
    sql = f"SELECT 1 FROM `{CDR}`.INFORMATION_SCHEMA.TABLES WHERE table_name = @t LIMIT 1"
    cfg = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("t", "STRING", table)])
    return next(client.query(sql, job_config=cfg).result(), None) is not None


def columns_of(table: str) -> set[str]:
    sql = f"SELECT column_name FROM `{CDR}`.INFORMATION_SCHEMA.COLUMNS WHERE table_name = @t"
    cfg = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ScalarQueryParameter("t", "STRING", table)])
    return {r.column_name for r in client.query(sql, job_config=cfg).result()}


COHORT_PARAM = bigquery.ArrayQueryParameter("ids", "INT64", COHORT_IDS)
QUERY_LOG: list[dict] = []


def run_query(sql: str, label: str, params=(), max_gb: float | None = None) -> pd.DataFrame:
    """Dry-run cost gate, then execute. Returns an aggregated DataFrame."""
    cap = MAX_QUERY_GB if max_gb is None else max_gb
    plist = [COHORT_PARAM, *params]
    dry = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=plist)
    gb = client.query(sql, job_config=dry).total_bytes_processed / 1e9
    usd = gb / 1024 * ON_DEMAND_USD_PER_TIB
    print(f"[{label}] dry run: {gb:,.1f} GB (~${usd:,.2f} at on-demand rates)")
    if gb > cap:
        raise RuntimeError(
            f"[{label}] would scan {gb:,.1f} GB, above MAX_QUERY_GB={cap:,.0f}. "
            f"Raise MAX_QUERY_GB if that is acceptable.")
    df = (client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=plist))
          .result().to_dataframe(create_bqstorage_client=False))
    QUERY_LOG.append({"label": label, "scanned_gb": round(gb, 1), "rows": len(df)})
    print(f"[{label}] {len(df):,} rows")
    return df


NEEDED = {"drug_exposure": DO_DRUGS, "concept_ancestor": DO_DRUGS,
          "measurement": DO_MEASUREMENTS, "observation": DO_SURVEYS,
          "condition_occurrence": DO_CONDITIONS, "concept": True}
AVAILABLE = {t: table_exists(t) for t in NEEDED}
print(pd.Series(AVAILABLE).rename("exists").to_string())
for t, wanted in NEEDED.items():
    if wanted and not AVAILABLE[t]:
        raise RuntimeError(f"`{CDR}.{t}` is missing but the matching domain is switched on")

## 4. PhecodeX — the panel we already test

Read from the existing table so the inventory is complete. A column is a case
for a participant when `select_phenotypes.py`'s `encode_binary` would call it 1;
eligibility is non-missing. The sex restriction from the PhecodeX definitions is
**not** re-applied here — `pheno_00_table_summary` owns that audit — so these
rows are an upper bound for sex-specific codes.

In [ ]:
TRUE_TOKENS = {"1", "1.0", "true", "t", "yes", "y", "case"}
FALSE_TOKENS = {"0", "0.0", "false", "f", "no", "n", "control"}


def encode_binary(s: pd.Series) -> pd.Series:
    """0/1/NA from the dialects the AoU phenotype tables use."""
    t = s.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=s.index, dtype=float)
    out[t.isin(TRUE_TOKENS)] = 1.0
    out[t.isin(FALSE_TOKENS)] = 0.0
    return out


def localize(uri: str) -> Path:
    if not uri.startswith("gs://"):
        p = Path(uri).expanduser()
        assert p.is_file(), f"{p} does not exist"
        return p.resolve()
    out = OUT_DIR / uri.rsplit("/", 1)[-1]
    if not out.exists():
        subprocess.check_call(["gsutil", "cp", uri, str(out)])
    return out.resolve()


phecode_rows = pd.DataFrame()
if DO_PHECODES and PHENOTYPE_CSV.strip():
    pheno = pd.read_csv(localize(PHENOTYPE_CSV), dtype=str, low_memory=False)
    pheno = pheno.drop(columns=[c for c in pheno.columns if c.startswith("Unnamed") or c == ""],
                       errors="ignore")
    assert "person_id" in pheno.columns, list(pheno.columns)[:8]
    pheno["person_id"] = pheno["person_id"].astype(str).str.strip()
    pheno = pheno.set_index("person_id")

    want = [str(i) for i in COHORT_IDS]
    have = [i for i in want if i in pheno.index]
    print(f"phenotype table covers {len(have):,} / {len(want):,} cohort ids")
    assert have, "no cohort id is present in the phenotype table — wrong table or wrong cohort"
    sub = pheno.loc[have]

    phe_cols = [c for c in sub.columns if c != "sex"]
    enc = sub[phe_cols].apply(encode_binary)
    phecode_rows = pd.DataFrame({
        "family": "PhecodeX disease",
        "trait_id": phe_cols,
        "trait_name": phe_cols,
        "trait_type": "binary",
        "n_eligible": enc.notna().sum().to_numpy(),
        "n_cases": (enc == 1.0).sum().to_numpy(),
    })
    print(f"{len(phecode_rows):,} phecode columns scored")
elif DO_PHECODES:
    print("INV_PHENOTYPE_CSV is blank — skipping the PhecodeX rows.\n"
          "Set it to a reachable copy of AoU_Phase2_Phenotype.csv.gz to fold the\n"
          "phecodes into this inventory; the other families do not depend on it.")

## 5. Prescriptions — `drug_exposure` rolled up to ATC classes

One binary exposure per ATC class: a case is a participant with at least one
drug exposure whose concept descends from that class. Everyone else in the
cohort is a control, so `n_eligible = n`. This is the *All by All* definition —
their GLP-1 analog phenotype is ATC `A10BJ`, an ATC 4th-level class.

In [ ]:
drug_rows = pd.DataFrame()
if DO_DRUGS:
    sql_drugs = f"""
    WITH cohort AS (SELECT id FROM UNNEST(@ids) AS id),
    exposures AS (
      SELECT DISTINCT d.person_id, d.drug_concept_id
      FROM `{CDR}.drug_exposure` d
      JOIN cohort ON cohort.id = d.person_id
      WHERE d.drug_concept_id IS NOT NULL AND d.drug_concept_id != 0
    )
    SELECT
      a.concept_code              AS trait_id,
      a.concept_name              AS trait_name,
      a.concept_class_id          AS atc_level,
      COUNT(DISTINCT e.person_id) AS n_cases
    FROM exposures e
    JOIN `{CDR}.concept_ancestor` ca ON ca.descendant_concept_id = e.drug_concept_id
    JOIN `{CDR}.concept` a ON a.concept_id = ca.ancestor_concept_id
    WHERE a.vocabulary_id = 'ATC' AND a.concept_class_id IN UNNEST(@levels)
    GROUP BY 1, 2, 3
    """
    drugs = run_query(sql_drugs, "drug_exposure -> ATC",
                      params=[bigquery.ArrayQueryParameter(
                          "levels", "STRING", list(ATC_LEVELS))])
    if len(drugs):
        drug_rows = pd.DataFrame({
            "family": "Prescription (ATC)",
            "trait_id": drugs["trait_id"],
            "trait_name": drugs["trait_name"],
            "trait_type": "binary",
            "n_eligible": N,
            "n_cases": drugs["n_cases"].astype(int),
            "subtype": drugs["atc_level"],
        })
        smaller = np.minimum(drug_rows["n_cases"], N - drug_rows["n_cases"])
        print(drug_rows.assign(smaller_arm=smaller)
              .groupby("subtype")
              .agg(classes=("trait_id", "size"),
                   testable=("smaller_arm", lambda s: int((s >= MIN_SAMPLES).sum())))
              .to_string())

## 6. Measurements — quantitative labs and physical measurements

A measurement concept is a candidate quantitative trait when enough participants
have at least one numeric value. The audit columns matter as much as the count:
`n_units` catches concepts whose values arrive in mixed units (the usual reason
an AoU lab is not usable as-is), and `n_distinct` catches concepts that are
numerically coded categories.

Classification is by vocabulary with a name-based override, and the crosstab is
printed so you can see what landed where — AoU physical measurements do not all
carry LOINC codes.

In [ ]:
PM_PATTERN = (r"\b(height|weight|body mass index|bmi|waist|hip|circumference|"
              r"blood pressure|systolic|diastolic|heart rate|pulse|wheelchair)\b")

measure_rows, units_audit = pd.DataFrame(), pd.DataFrame()
if DO_MEASUREMENTS:
    sql_meas = f"""
    WITH cohort AS (SELECT id FROM UNNEST(@ids) AS id),
    vals AS (
      SELECT m.person_id, m.measurement_concept_id AS cid, m.value_as_number AS v,
             m.unit_concept_id AS uid
      FROM `{CDR}.measurement` m
      JOIN cohort ON cohort.id = m.person_id
      WHERE m.value_as_number IS NOT NULL
        AND NOT IS_NAN(m.value_as_number)
        AND m.measurement_concept_id IS NOT NULL
        AND m.measurement_concept_id != 0
    )
    SELECT
      v.cid                                  AS concept_id,
      c.concept_name                         AS trait_name,
      c.vocabulary_id                        AS vocabulary,
      COUNT(DISTINCT v.person_id)            AS n_measured,
      COUNT(*)                               AS n_rows,
      COUNT(DISTINCT v.v)                    AS n_distinct,
      COUNT(DISTINCT v.uid)                  AS n_units,
      APPROX_QUANTILES(v.v, 100)[OFFSET(5)]  AS p05,
      APPROX_QUANTILES(v.v, 100)[OFFSET(50)] AS p50,
      APPROX_QUANTILES(v.v, 100)[OFFSET(95)] AS p95
    FROM vals v
    JOIN `{CDR}.concept` c ON c.concept_id = v.cid
    GROUP BY 1, 2, 3
    HAVING n_measured >= @floor
    """
    meas = run_query(sql_meas, "measurement -> quantitative",
                     params=[bigquery.ScalarQueryParameter("floor", "INT64", MIN_SAMPLES)])
    if len(meas):
        is_pm = meas["trait_name"].fillna("").str.contains(PM_PATTERN, case=False, regex=True)
        meas["family"] = np.where(
            is_pm, "Physical measurement",
            np.where(meas["vocabulary"].eq("LOINC"), "Lab measurement", "Other measurement"))
        measure_rows = pd.DataFrame({
            "family": meas["family"],
            "trait_id": meas["concept_id"].astype(str),
            "trait_name": meas["trait_name"],
            "trait_type": "quantitative",
            "n_eligible": meas["n_measured"].astype(int),
            "n_cases": np.nan,
            "n_distinct": meas["n_distinct"].astype(int),
            "n_units": meas["n_units"].astype(int),
            "subtype": meas["vocabulary"],
        })
        units_audit = meas[["concept_id", "trait_name", "family", "vocabulary", "n_measured",
                            "n_rows", "n_units", "n_distinct", "p05", "p50", "p95"]].copy()
        units_audit["rows_per_person"] = (
            units_audit["n_rows"] / units_audit["n_measured"]).round(1)
        units_audit = units_audit.sort_values("n_measured", ascending=False)
        print(meas.groupby(["family", "vocabulary"]).size().rename("concepts").to_string())
        print(f"\nconcepts with more than one unit: "
              f"{int((meas['n_units'] > 1).sum()):,} of {len(meas):,}")

## 7. Surveys — `observation` as question x answer binaries

A self-reported binary trait is one answer option to one survey question: cases
answered that option, controls answered the question differently. Participants
who skipped, declined, or answered "don't know" are dropped from **both** arms —
they are not controls — so `n_eligible` is question-specific and below `n`.

In [ ]:
NONANSWER = (r"skip|prefer not to answer|don'?t know|do not know|not sure|^pmi[_ ]")

survey_rows = pd.DataFrame()
if DO_SURVEYS:
    sql_survey = f"""
    WITH cohort AS (SELECT id FROM UNNEST(@ids) AS id),
    answers AS (
      SELECT DISTINCT
        o.person_id,
        COALESCE(NULLIF(o.observation_source_concept_id, 0),
                 o.observation_concept_id) AS qid,
        o.value_source_concept_id          AS aid
      FROM `{CDR}.observation` o
      JOIN cohort ON cohort.id = o.person_id
      WHERE o.value_source_concept_id IS NOT NULL AND o.value_source_concept_id != 0
    )
    SELECT
      a.qid, q.concept_name AS question, q.vocabulary_id AS q_vocab,
      a.aid, v.concept_name AS answer,
      COUNT(DISTINCT a.person_id) AS n_cases
    FROM answers a
    JOIN `{CDR}.concept` q ON q.concept_id = a.qid
    JOIN `{CDR}.concept` v ON v.concept_id = a.aid
    WHERE q.vocabulary_id = 'PPI'
    GROUP BY 1, 2, 3, 4, 5
    """
    sv = run_query(sql_survey, "observation -> survey answers")
    if len(sv):
        real = ~sv["answer"].fillna("").str.contains(NONANSWER, case=False, regex=True)
        dropped = int((~real).sum())
        sv = sv[real].copy()
        # Denominator: participants who gave a real answer to that question.
        sv["n_eligible"] = sv.groupby("qid")["n_cases"].transform("sum")
        survey_rows = pd.DataFrame({
            "family": "Self-reported (survey)",
            "trait_id": sv["qid"].astype(str) + ":" + sv["aid"].astype(str),
            "trait_name": sv["question"].fillna("").str.slice(0, 120) + " = "
                          + sv["answer"].fillna("").astype(str),
            "trait_type": "binary",
            "n_eligible": sv["n_eligible"].astype(int),
            "n_cases": sv["n_cases"].astype(int),
            "subtype": sv["q_vocab"],
        })
        print(f"{dropped:,} skip / decline / don't-know options dropped from both arms")
        print(f"{sv['qid'].nunique():,} questions, {len(survey_rows):,} question x answer pairs")
        print("note: n_eligible sums distinct responders per option, so a multi-select\n"
              "question inflates it — check such questions before testing them.")

## 8. Conditions — `condition_occurrence` SNOMED concepts (optional)

Off by default: these overlap the PhecodeX panel almost completely and would
double-count in the family totals. Switch `DO_CONDITIONS` on if you want to see
how much the phecode roll-up loses relative to raw condition concepts.

In [ ]:
condition_rows = pd.DataFrame()
if DO_CONDITIONS:
    sql_cond = f"""
    WITH cohort AS (SELECT id FROM UNNEST(@ids) AS id),
    occ AS (
      SELECT DISTINCT co.person_id, co.condition_concept_id AS cid
      FROM `{CDR}.condition_occurrence` co
      JOIN cohort ON cohort.id = co.person_id
      WHERE co.condition_concept_id IS NOT NULL AND co.condition_concept_id != 0
    )
    SELECT o.cid AS concept_id, c.concept_name AS trait_name, c.vocabulary_id AS vocabulary,
           COUNT(DISTINCT o.person_id) AS n_cases
    FROM occ o JOIN `{CDR}.concept` c ON c.concept_id = o.cid
    GROUP BY 1, 2, 3
    HAVING n_cases >= @floor
    """
    cond = run_query(sql_cond, "condition_occurrence -> SNOMED",
                     params=[bigquery.ScalarQueryParameter("floor", "INT64", MIN_SAMPLES)])
    if len(cond):
        condition_rows = pd.DataFrame({
            "family": "Condition (SNOMED)",
            "trait_id": cond["concept_id"].astype(str),
            "trait_name": cond["trait_name"],
            "trait_type": "binary",
            "n_eligible": N,
            "n_cases": cond["n_cases"].astype(int),
            "subtype": cond["vocabulary"],
        })

## 9. Combine and apply the floor

`n_eff` puts the families on one scale: `4 * n_cases * (1 - phi)` is the
effective sample size of a 1-df score test on a binary trait with case fraction
phi, against `n_measured` for a quantitative trait on everyone measured.

In [ ]:
inv = pd.concat([df for df in (phecode_rows, drug_rows, measure_rows,
                               survey_rows, condition_rows) if len(df)],
                ignore_index=True)
assert len(inv), "no candidate phenotypes were assembled — every domain was off or empty"

for c in ("subtype", "n_distinct", "n_units"):
    if c not in inv.columns:
        inv[c] = np.nan

binary = inv["trait_type"].eq("binary")
inv["n_controls"] = np.where(binary, inv["n_eligible"] - inv["n_cases"].fillna(0), np.nan)
inv["n_min_arm"] = np.where(binary, np.minimum(inv["n_cases"], inv["n_controls"]),
                            inv["n_eligible"])
inv["prevalence"] = np.where(
    binary, inv["n_cases"] / inv["n_eligible"].where(inv["n_eligible"] > 0), np.nan)
inv["coverage"] = inv["n_eligible"] / N
inv["n_eff"] = np.where(binary,
                        4 * inv["n_cases"].fillna(0) * (1 - inv["prevalence"].fillna(0)),
                        inv["n_eligible"])
inv["testable"] = np.where(
    binary,
    inv["n_min_arm"] >= MIN_SAMPLES,
    (inv["n_eligible"] >= MIN_SAMPLES) & (inv["n_distinct"].fillna(0) >= MIN_DISTINCT))

inv = inv.sort_values(["family", "n_eff"], ascending=[True, False]).reset_index(drop=True)
inv = inv[["family", "subtype", "trait_id", "trait_name", "trait_type", "n_eligible",
           "n_cases", "n_controls", "n_min_arm", "prevalence", "coverage", "n_distinct",
           "n_units", "n_eff", "testable"]]

TESTABLE_COL = f"testable_{MIN_SAMPLES}"
by_family = pd.DataFrame({
    "candidates": inv.groupby("family").size(),
    TESTABLE_COL: inv.groupby("family")["testable"].sum().astype(int),
    "median_n_eff_testable": inv[inv["testable"]].groupby("family")["n_eff"].median(),
    "max_n_eff": inv.groupby("family")["n_eff"].max(),
}).sort_values(TESTABLE_COL, ascending=False)
by_family.loc["Total"] = by_family.sum(numeric_only=True)
by_family.loc["Total", "median_n_eff_testable"] = float(inv.loc[inv["testable"], "n_eff"].median())
by_family.loc["Total", "max_n_eff"] = float(inv["n_eff"].max())

print(by_family.round(0).astype({"candidates": int, TESTABLE_COL: int}).to_string())
print(f"\ntotal testable at the {MIN_SAMPLES}-sample floor: "
      f"{int(inv['testable'].sum()):,} of {len(inv):,} candidates")

## 10. How the count moves with the floor

In [ ]:
THR = np.arange(1, 1001)
families = list(inv["family"].unique())


def curve(values) -> np.ndarray:
    return (np.asarray(values, float)[None, :] >= THR[:, None]).sum(axis=1)


cutoff = pd.DataFrame({"cutoff": THR})
for fam in families:
    d = inv[inv["family"].eq(fam)]
    usable = d["n_min_arm"].to_numpy(float)
    if d["trait_type"].iloc[0] == "quantitative":
        usable = np.where(d["n_distinct"].fillna(0).to_numpy() >= MIN_DISTINCT, usable, 0.0)
    cutoff[fam] = curve(usable)
cutoff["All families"] = cutoff[families].sum(axis=1)

print("testable traits at each floor (rows = floor on the smaller arm / n measured):")
print(cutoff.set_index("cutoff").loc[[10, 20, MIN_SAMPLES, 100, 200, 500]].to_string())

## 11. Figure

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PALETTE = {"PhecodeX disease": "#4C72B0", "Prescription (ATC)": "#8B2F3F",
           "Self-reported (survey)": "#4FA67F", "Lab measurement": "#D1804A",
           "Physical measurement": "#E0A64B", "Other measurement": "#9C9C9C",
           "Condition (SNOMED)": "#6A5ACD"}
order = [f for f in by_family.index if f != "Total"]
colors = [PALETTE.get(f, "#9C9C9C") for f in order]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 4.0))

vals = by_family.loc[order, TESTABLE_COL].to_numpy(float)
axA.barh(order[::-1], vals[::-1], color=colors[::-1], height=0.62)
for y, v in enumerate(vals[::-1]):
    axA.text(v + max(vals) * 0.015, y, f"{int(v):,}", va="center", fontsize=8)
axA.set_xlabel(f"Traits testable at a {MIN_SAMPLES}-sample floor")
axA.set_title(f"{int(inv['testable'].sum()):,} testable traits in a cohort of {N:,}", loc="left")
axA.set_xlim(0, max(vals) * 1.18)
axA.tick_params(axis="y", length=0)

for fam, col in zip(order, colors):
    axB.plot(cutoff["cutoff"], cutoff[fam], color=col, lw=1.4, label=fam)
axB.axvline(MIN_SAMPLES, color="#555555", lw=1.0, ls=":")
axB.set_xscale("log")
axB.set_yscale("log")
axB.set_ylim(0.8, max(vals) * 2.0)
axB.text(MIN_SAMPLES * 1.12, 1.0, f"floor = {MIN_SAMPLES}",
         fontsize=8, color="#555555", va="bottom")
axB.set_xlabel("Floor on the smaller arm / samples measured")
axB.set_ylabel("Traits testable")
axB.set_title("Each family's count against the floor", loc="left")
axB.legend(frameon=False, fontsize=7, loc="upper right")

for ax in (axA, axB):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

fig.tight_layout()
FIG_PATH = OUT_DIR / "phenotype_inventory.png"
fig.savefig(FIG_PATH, dpi=200)
print("wrote", FIG_PATH)

## 12. Write out

Two copies of the inventory: the full table for work inside the workspace, and
an export copy with any count below `MIN_CELL` blanked, which is the one to move
out. Nothing person-level is written by either.

In [ ]:
INV_PATH = OUT_DIR / "phenotype_inventory.tsv"
EXPORT_PATH = OUT_DIR / "phenotype_inventory_export.tsv"
inv.to_csv(INV_PATH, sep="\t", index=False)

export = inv.copy()
small = (export["n_eligible"].lt(MIN_CELL)
         | export["n_cases"].lt(MIN_CELL).fillna(False)
         | export["n_controls"].lt(MIN_CELL).fillna(False))
for c in ("n_eligible", "n_cases", "n_controls", "n_min_arm", "prevalence", "coverage", "n_eff"):
    export.loc[small, c] = np.nan
export.to_csv(EXPORT_PATH, sep="\t", index=False)
print(f"{int(small.sum()):,} rows had a count below {MIN_CELL} and are blanked in the export copy")

by_family.to_csv(OUT_DIR / "inventory_by_family.tsv", sep="\t")
cutoff.to_csv(OUT_DIR / "inventory_cutoff_curve.tsv", sep="\t", index=False)
if len(units_audit):
    units_audit.to_csv(OUT_DIR / "inventory_units_audit.tsv", sep="\t", index=False)
pd.DataFrame(QUERY_LOG).to_csv(OUT_DIR / "query_log.tsv", sep="\t", index=False)

cut = cutoff.set_index("cutoff")
lines = [
    f"# Phenotype inventory at a {MIN_SAMPLES}-sample floor",
    "",
    f"* CDR: `{CDR}`",
    f"* cohort: `{COHORT_SRC}`, n = {N:,}",
    f"* rule: binary needs {MIN_SAMPLES} in the smaller arm; quantitative needs "
    f"{MIN_SAMPLES} measured and {MIN_DISTINCT} distinct values",
    f"* BigQuery scanned: {sum(q['scanned_gb'] for q in QUERY_LOG):,.0f} GB across "
    f"{len(QUERY_LOG)} queries",
    "",
    "## Testable traits by family",
    "",
]
for fam in order:
    row = by_family.loc[fam]
    lines.append(f"* {fam}: **{int(row[TESTABLE_COL]):,}** of {int(row['candidates']):,} "
                 f"candidates, median n_eff {row['median_n_eff_testable']:,.0f}")
lines += [
    "",
    f"* total testable: **{int(inv['testable'].sum()):,}**",
    f"* at a floor of 100 instead of {MIN_SAMPLES}: {int(cut.loc[100, 'All families']):,}",
    f"* at a floor of 200: {int(cut.loc[200, 'All families']):,}",
]
(OUT_DIR / "summary.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

print("\nfiles written:")
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name}  ({p.stat().st_size / 1e3:,.1f} kB)")

## 13. What to do with it

`phenotype_inventory.tsv` sorted by `n_eff` within family is the shortlist. Two
checks before promoting any row into the scan:

- **Measurements**: `n_units > 1` means values arrive in mixed units and need
  harmonising before the concept is a trait; `rows_per_person` in
  `inventory_units_audit.tsv` tells you whether one value per person is a median
  over many draws or a single reading.
- **Surveys**: a multi-select question inflates `n_eligible`, because each option
  is counted among the responders. Confirm the question is single-select before
  treating the complement as controls.

Running quantitative traits through FELIX needs two changes that are not in the
pipeline yet: `scripts/fit_felix_null.R` does not pass `--invNormalize` to SAIGE
(needed for skewed measurements), and `scripts/select_phenotypes.py` is
binary-only (`encode_binary`, `--min-cases`, ranking by the smaller arm). The
`trait_type` input is already plumbed through `FelixGenome.wdl` and
`FitFelixNulls.wdl`, so no WDL change is required.